
# Import Libraries


In [3]:
import numpy as np
from decision_tree import DecisionTreeRegressor
from Bagging import BaggingRegressor
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
import time


In [6]:
import os
import pandas as pd

#print('cwd:', os.getcwd())
#print('files:', os.listdir())

train_df = pd.read_csv('train.csv')
value_df = pd.read_csv('val.csv')
test_df = pd.read_csv('test.csv')

def prepare_data(df):
    X = df.drop(columns=['Calories']).values  # Chuyển về numpy array
    y = df['Calories'].values                 # Chuyển về numpy array
    return X, y

X_train, y_train = prepare_data(train_df)
X_val, y_val = prepare_data(value_df)

print(f"Training trên tập dữ liệu kích thước: {X_train.shape}")


Training trên tập dữ liệu kích thước: (9600, 7)


## Train 50 cây, độ sâu 10, min_samples_split=10

# Bagging Reggressor from scratch

In [11]:
# Khởi tạo Bagging Regressor
# - n_estimator: số lượng (Tăng lên 10-20 để chính xác hơn nhưng lâu hơn)
# - max_depth: Độ sâu cây tối đa (Tăng lên 10 nếu muốn cây học kỹ hơn)
model = BaggingRegressor(base_tree_cls=DecisionTreeRegressor, 
                         n_estimators=50, 
                         min_samples_split=10, 
                         max_depth=10)
start=time.time()
model.fit(X_train, y_train)
manual_time=time.time()-start
print(f"Thời gian huấn luyện: {manual_time:.2f} giây")
print("Huấn luyện hoàn tất!")

# ==========================================
# PHẦN 4: ĐÁNH GIÁ & VẼ BIỂU ĐỒ
# ==========================================


# Dự đoán trên tập test
y_pred = model.predict(X_val)

# Tính toán sai số
manual_mse = mean_squared_error(y_val, y_pred)
manual_rmse = np.sqrt(manual_mse)
manual_r2 = r2_score(y_val, y_pred)

print("-" * 30)
print(f"Kết quả đánh giá:")
print(f"RMSE (Sai số trung bình): {manual_rmse:.2f} Calories")
print(f"R2 Score (Độ phù hợp): {manual_r2:.4f}")
print("-" * 30)

'''
# Vẽ biểu đồ So sánh Thực tế vs Dự đoán
plt.figure(figsize=(10, 6))
# Vẽ 100 điểm đầu tiên cho dễ nhìn
plt.plot(y_val[:100], label='Thực tế (Actual)', color='blue', marker='o')
plt.plot(y_pred[:100], label='Dự đoán (Predicted)', color='red', linestyle='--', marker='x')
plt.title('So sánh Calo Thực tế và Dự đoán (100 mẫu đầu)')
plt.xlabel('Mẫu số')
plt.ylabel('Calories')
plt.legend()
plt.grid(True)
plt.show()

# Vẽ biểu đồ phân tán (Scatter Plot)
plt.figure(figsize=(8, 8))
plt.scatter(y_val, y_pred, alpha=0.5, color='purple')
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'k--', lw=2) # Đường chéo chuẩn
plt.xlabel('Thực tế')
plt.ylabel('Dự đoán')
plt.title('Scatter Plot: Thực tế vs Dự đoán')
plt.show()
''';

Thời gian huấn luyện: 1712.21 giây
Huấn luyện hoàn tất!
------------------------------
Kết quả đánh giá:
RMSE (Sai số trung bình): 3.29 Calories
R2 Score (Độ phù hợp): 0.9972
------------------------------


# Bagging Reggressor using sklearn

# Import Libraries for BaggingRegressor

In [12]:
from sklearn.ensemble import BaggingRegressor as SklearnBaggingRegressor
from sklearn.tree import DecisionTreeRegressor as SklearnDecisionTreeRegressor

In [13]:
'''
model = SklearnBaggingRegressor(estimator=SklearnDecisionTreeRegressor(max_depth=10),
                         n_estimators=20,
                         random_state=42)
'''
my_tree = SklearnDecisionTreeRegressor(
    max_depth=10,            
    min_samples_split=10,   
    random_state=42
)


model = SklearnBaggingRegressor(
    estimator=my_tree,      #
    n_estimators=50,
    random_state=42,
    
)

start=time.time()
model.fit(X_train, y_train)
sklearn_time=time.time()-start
print(f"Thời gian huấn luyện sklearn: {sklearn_time:.2f} giây")
print("Huấn luyện sklearn hoàn tất!")
y_pred = model.predict(X_val)

sklearn_mse = mean_squared_error(y_val, y_pred)
sklearn_rmse = np.sqrt(sklearn_mse)
sklearn_r2 = r2_score(y_val, y_pred)

print("-" * 30)
print(f"Kết quả đánh giá của mô hình lấy từ thư viện:")
print(f"RMSE (Sai số trung bình): {sklearn_rmse:.2f} Calories")
print(f"R2 Score (Độ phù hợp): {sklearn_r2:.4f}")
print("-" * 30)


'''
plt.figure(figsize=(10, 6))
# Vẽ 100 điểm đầu tiên cho dễ nhìn
plt.plot(y_val[:100], label='Thực tế (Actual)', color='blue', marker='o')
plt.plot(y_pred[:100], label='Dự đoán (Predicted)', color='red', linestyle='--', marker='x')
plt.title('So sánh Calo Thực tế và Dự đoán (100 mẫu đầu)')
plt.xlabel('Mẫu số')
plt.ylabel('Calories')
plt.legend()
plt.grid(True)
plt.show()

# Vẽ biểu đồ phân tán (Scatter Plot)
plt.figure(figsize=(8, 8))
plt.scatter(y_val, y_pred, alpha=0.5, color='green')
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'k--', lw=2) # Đường chéo chuẩn
plt.xlabel('Thực tế')
plt.ylabel('Dự đoán')
plt.title('Scatter Plot: Thực tế vs Dự đoán (sklearn)')
plt.show()

''';

Thời gian huấn luyện sklearn: 1.64 giây
Huấn luyện sklearn hoàn tất!
------------------------------
Kết quả đánh giá của mô hình lấy từ thư viện:
RMSE (Sai số trung bình): 3.79 Calories
R2 Score (Độ phù hợp): 0.9962
------------------------------


In [14]:
results = [
    ["Manual", manual_time, manual_rmse, manual_r2],
    ["Sklearn", sklearn_time, sklearn_rmse, sklearn_r2]
]

df = pd.DataFrame(results, columns=["Model Type", "Time (s)", "RMSE", "R2 Score"])
display(df)

,Model Type,Time (s),RMSE,R2 Score
0,Manual,1712.207910,3.294264,0.997162
1,Sklearn,1.636871,3.792899,0.996237
